In [1]:
import numpy as np
import pandas as pd

In [2]:

file_path = 'C:\\Users\\levm\\Desktop\\‏‏data_set.csv'


df = pd.read_csv(file_path, encoding="cp1255")
df.head()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12737 entries, 0 to 12736
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   question        12737 non-null  str    
 1   teacher_answer  12735 non-null  str    
 2   student_answer  12734 non-null  str    
 3   score           12723 non-null  float64
dtypes: float64(1), str(3)
memory usage: 4.3 MB


In [3]:
df = df.dropna().reset_index(drop=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12722 entries, 0 to 12721
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   question        12722 non-null  str    
 1   teacher_answer  12722 non-null  str    
 2   student_answer  12722 non-null  str    
 3   score           12722 non-null  float64
dtypes: float64(1), str(3)
memory usage: 4.3 MB


In [4]:
df["text"] = (
    "[Q] " + df["question"] +
    " [T] " + df["teacher_answer"] +
    " [S] " + df["student_answer"]
)

In [5]:
df_model = df[["text", "score"]]
df_model.head()

,text,score
0,[Q] מה היו תוצאות מלחמת העולם הראשונה? [T] סיו...,1.0
1,[Q] מה היו תוצאות מלחמת העולם הראשונה? [T] סיו...,0.9
2,[Q] מה היו תוצאות מלחמת העולם הראשונה? [T] סיו...,0.7
3,[Q] מה היו תוצאות מלחמת העולם הראשונה? [T] סיו...,0.4
4,[Q] מה היו תוצאות מלחמת העולם הראשונה? [T] סיו...,0.1


In [125]:
df_model.info()

<class 'pandas.DataFrame'>
RangeIndex: 12722 entries, 0 to 12721
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   text            12722 non-null  str     
 1   score           12722 non-null  float64 
 2   text_len_chars  12722 non-null  int64   
 3   text_len_words  12722 non-null  int64   
 4   score_bin       12722 non-null  category
dtypes: category(1), float64(1), int64(2), str(1)
memory usage: 4.5 MB


In [99]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [12]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("microsoft/mdeberta-v3-base")
model = AutoModel.from_pretrained("microsoft/mdeberta-v3-base")

C:\Users\levm\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

KeyboardInterrupt



In [126]:
df_model.head()



,text,score,text_len_chars,text_len_words,score_bin
0,[Q] מה היו תוצאות מלחמת העולם הראשונה? [T] סיו...,1.0,489,73,"(0.8, 1.0]"
1,[Q] מה היו תוצאות מלחמת העולם הראשונה? [T] סיו...,0.9,426,63,"(0.8, 1.0]"
2,[Q] מה היו תוצאות מלחמת העולם הראשונה? [T] סיו...,0.7,400,59,"(0.6, 0.8]"
3,[Q] מה היו תוצאות מלחמת העולם הראשונה? [T] סיו...,0.4,346,51,"(0.2, 0.4]"
4,[Q] מה היו תוצאות מלחמת העולם הראשונה? [T] סיו...,0.1,294,43,"(-0.001, 0.2]"


In [127]:
df_model.columns.tolist()

['text', 'score', 'text_len_chars', 'text_len_words', 'score_bin']

In [128]:
df_model["score"].describe()


count    12722.000000
mean         0.608019
std          0.341373
min          0.000000
25%          0.300000
50%          0.700000
75%          0.950000
max          1.000000
Name: score, dtype: float64

In [129]:
print(df_model.iloc[0]["text"][:500])


[Q] מה היו תוצאות מלחמת העולם הראשונה? [T] סיום ב-1918, פירוק האימפריה האוסטרו-הונגרית והעות'מאנית, חוזה ורסאי, פיצויים לגרמניה, איבוד שטחים, הקמת מדינות חדשות, משבר כלכלי, אבדות רבות, חוסר יציבות פוליטית ועליית משטרים קיצוניים [S] מלחמת העולם הראשונה הסתיימה בשנת 1918 והשאירה את אירופה במצב שונה לגמרי מזה שהיה לפני המלחמה. האימפריות האוסטרו-הונגרית והעות'מאנית התפרקו, וגרמניה נאלצה לשלם פיצויים כבדים ולאבד שטחים. בנוסף קמו מדינות חדשות והמצב הכלכלי והפוליטי במדינות רבות הפך ללא יציב.


In [105]:

# Drop missing values just in case
df = df.dropna(subset=["score"]).reset_index(drop=True)

# Define score bins
bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
labels = ["0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]

# Create a binned column
df["score_bin"] = pd.cut(df["score"], bins=bins, labels=labels, include_lowest=True)

# Count how many samples in each bin
score_counts = df["score_bin"].value_counts().sort_index()

# Print results
print("Score distribution:")
print(score_counts)
print("\nPercent distribution:")
print((score_counts / len(df) * 100).round(2))

Score distribution:
score_bin
0-0.2      2756
0.2-0.4    1639
0.4-0.6    1398
0.6-0.8    1942
0.8-1.0    4987
Name: count, dtype: int64

Percent distribution:
score_bin
0-0.2      21.66
0.2-0.4    12.88
0.4-0.6    10.99
0.6-0.8    15.26
0.8-1.0    39.20
Name: count, dtype: float64


In [106]:

# Remove missing values
df = df.dropna(subset=["question", "teacher_answer", "student_answer", "score"])

# Character lengths
df["question_len"] = df["question"].astype(str).str.len()
df["teacher_len"] = df["teacher_answer"].astype(str).str.len()
df["student_len"] = df["student_answer"].astype(str).str.len()

print("=" * 60)
print("QUESTION LENGTHS")
print(df["question_len"].describe())

print("\n" + "=" * 60)
print("TEACHER ANSWER LENGTHS")
print(df["teacher_len"].describe())

print("\n" + "=" * 60)
print("STUDENT ANSWER LENGTHS")
print(df["student_len"].describe())

# Length categories based on quartiles
q25 = df["student_len"].quantile(0.25)
q75 = df["student_len"].quantile(0.75)

print("\n" + "=" * 60)
print(f"Short threshold  : <= {q25:.0f} chars")
print(f"Long threshold   : >= {q75:.0f} chars")

df["length_group"] = "medium"
df.loc[df["student_len"] <= q25, "length_group"] = "short"
df.loc[df["student_len"] >= q75, "length_group"] = "long"

print("\n" + "=" * 60)
print("LENGTH DISTRIBUTION")
print(df["length_group"].value_counts())

print("\nPERCENTAGES")
print((df["length_group"].value_counts(normalize=True) * 100).round(2))

QUESTION LENGTHS
count    12722.000000
mean        29.600299
std         16.626597
min          6.000000
25%         18.000000
50%         26.000000
75%         37.000000
max         92.000000
Name: question_len, dtype: float64

TEACHER ANSWER LENGTHS
count    12722.000000
mean        76.080805
std         48.901323
min          4.000000
25%         43.000000
50%         62.000000
75%         92.000000
max        308.000000
Name: teacher_len, dtype: float64

STUDENT ANSWER LENGTHS
count    12722.000000
mean        73.338312
std         66.721915
min          2.000000
25%         26.000000
50%         43.000000
75%        112.000000
max        403.000000
Name: student_len, dtype: float64

Short threshold  : <= 26 chars
Long threshold   : >= 112 chars

LENGTH DISTRIBUTION
length_group
medium    6193
short     3329
long      3200
Name: count, dtype: int64

PERCENTAGES
length_group
medium    48.68
short     26.17
long      25.15
Name: proportion, dtype: float64


In [107]:

# ==========================
# STUDENT ANSWER LENGTH
# ==========================
df["answer_length"] = df["student_answer"].astype(str).apply(lambda x: len(x.split()))

# ==========================
# LENGTH GROUPS
# ==========================
df["length_group"] = pd.qcut(
    df["answer_length"],
    q=3,
    labels=["short", "medium", "long"]
)

# ==========================
# SCORE STATS PER GROUP
# ==========================
result = df.groupby("length_group")["score"].agg([
    "count",
    "mean",
    "median",
    "min",
    "max"
])

print("\n" + "=" * 60)
print("SCORE VS ANSWER LENGTH")
print(result)

# ==========================
# HIGH SCORE RATE
# ==========================
high_score = (
    df.groupby("length_group")
      .apply(lambda x: (x["score"] >= 0.8).mean() * 100)
)

print("\n" + "=" * 60)
print("PERCENT OF ANSWERS WITH SCORE >= 0.8")
print(high_score.round(2))

# ==========================
# CORRELATION
# ==========================
corr = df["answer_length"].corr(df["score"])

print("\n" + "=" * 60)
print(f"CORRELATION(length, score) = {corr:.4f}")


SCORE VS ANSWER LENGTH
              count      mean  median  min  max
length_group                                   
short          4298  0.626333     0.8  0.0  1.0
medium         4223  0.609538     0.7  0.0  1.0
long           4201  0.587755     0.6  0.0  1.0

PERCENT OF ANSWERS WITH SCORE >= 0.8
length_group
short     50.77
medium    46.72
long      37.61
dtype: float64

CORRELATION(length, score) = -0.0129


In [108]:


df = df.dropna()

df["answer_len"] = (
    df["student_answer"]
    .astype(str)
    .str.split()
    .str.len()
)

df["length_group"] = pd.cut(
    df["answer_len"],
    bins=[0, 20, 80, 100000],
    labels=["short", "medium", "long"]
)

df["score_group"] = pd.cut(
    df["score"],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=["0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"],
    include_lowest=True
)

table = pd.crosstab(
    df["length_group"],
    df["score_group"]
)

print("\nCOUNT TABLE")
print(table)

print("\nPERCENT TABLE")
print(round(table / len(df) * 100, 2))


COUNT TABLE
score_group   0-0.2  0.2-0.4  0.4-0.6  0.6-0.8  0.8-1.0
length_group                                           
short          2284     1200     1006     1079     4192
medium          472      439      392      863      795

PERCENT TABLE
score_group   0-0.2  0.2-0.4  0.4-0.6  0.6-0.8  0.8-1.0
length_group                                           
short         17.95     9.43     7.91     8.48    32.95
medium         3.71     3.45     3.08     6.78     6.25


In [109]:


# Answer length (word count)
df["answer_len"] = (
    df["student_answer"]
    .astype(str)
    .str.split()
    .str.len()
)

# Length categories
df["length_group"] = pd.cut(
    df["answer_len"],
    bins=[0, 20, 80, 100000],
    labels=["short", "medium", "long"]
)

# Score categories
df["score_group"] = pd.cut(
    df["score"],
    bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=["0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"],
    include_lowest=True
)

# Count combinations
table = pd.crosstab(
    df["length_group"],
    df["score_group"]
)

print("=" * 60)
print("COUNT TABLE")
print(table)

# Convert to long format
analysis = (
    table.stack()
    .reset_index()
)

analysis.columns = ["length_group", "score_group", "count"]

# Calculate percentage of entire dataset
analysis["percent_of_dataset"] = (
    analysis["count"] / len(df) * 100
).round(2)

# Average cell size
average_count = analysis["count"].mean()

# Mark underrepresented groups
analysis["underrepresented"] = analysis["count"] < (average_count * 0.5)

print("\n" + "=" * 60)
print("GROUP ANALYSIS")
print(
    analysis.sort_values("count")
)

print("\n" + "=" * 60)
print("UNDERREPRESENTED GROUPS")
print(
    analysis[analysis["underrepresented"]]
    .sort_values("count")
)

print("\nAverage samples per group:", round(average_count))

COUNT TABLE
score_group   0-0.2  0.2-0.4  0.4-0.6  0.6-0.8  0.8-1.0
length_group                                           
short          2284     1200     1006     1079     4192
medium          472      439      392      863      795

GROUP ANALYSIS
  length_group score_group  count  percent_of_dataset  underrepresented
7       medium     0.4-0.6    392                3.08              True
6       medium     0.2-0.4    439                3.45              True
5       medium       0-0.2    472                3.71              True
9       medium     0.8-1.0    795                6.25             False
8       medium     0.6-0.8    863                6.78             False
2        short     0.4-0.6   1006                7.91             False
3        short     0.6-0.8   1079                8.48             False
1        short     0.2-0.4   1200                9.43             False
0        short       0-0.2   2284               17.95             False
4        short     0.8-1.0  

In [51]:


data = {
    'length_group': ['short', 'short', 'short', 'short', 'short',
                     'medium', 'medium', 'medium', 'medium', 'medium'],
    'score_group': ['0-0.2', '0.2-0.4', '0.4-0.6', '0.6-0.8', '0.8-1.0',
                    '0-0.2', '0.2-0.4', '0.4-0.6', '0.6-0.8', '0.8-1.0'],
    'count': [1954, 839, 972, 1079, 4192, 86, 5, 42, 53, 776]
}

df_counts = pd.DataFrame(data)

avg_group_size = df_counts['count'].mean()
print(f"Average samples per group: {avg_group_size:.0f}")

df_counts['underrepresented'] = df_counts['count'] < avg_group_size

df_counts['samples_to_add'] = df_counts.apply(
    lambda row: max(0, int(avg_group_size - row['count'])) if row['underrepresented'] else 0,
    axis=1
)

underrep_groups = df_counts[df_counts['underrepresented']].copy()

print("UNDERREPRESENTED GROUPS AND SAMPLES TO ADD")
print(underrep_groups[['length_group', 'score_group', 'count', 'samples_to_add']])

Average samples per group: 1000
UNDERREPRESENTED GROUPS AND SAMPLES TO ADD
  length_group score_group  count  samples_to_add
1        short     0.2-0.4    839             160
2        short     0.4-0.6    972              27
5       medium       0-0.2     86             913
6       medium     0.2-0.4      5             994
7       medium     0.4-0.6     42             957
8       medium     0.6-0.8     53             946
9       medium     0.8-1.0    776             223


In [57]:

# ==========================
# LOAD DATASET
# ==========================

import numpy as np
# ==========================
# WORD COUNT
# ==========================
df["word_count"] = (
    df["student_answer"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

# ==========================
# LENGTH GROUPS
# ==========================
q1 = df["word_count"].quantile(0.33)
q2 = df["word_count"].quantile(0.66)

df["length_group"] = pd.cut(
    df["word_count"],
    bins=[-1, q1, q2, np.inf],
    labels=["short", "medium", "long"]
)

# ==========================
# SCORE GROUPS
# ==========================
df["score_group"] = pd.cut(
    df["score"],
    bins=[-0.01, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=["0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]
)

# ==========================
# COUNT TABLE
# ==========================
table = pd.crosstab(
    df["length_group"],
    df["score_group"]
)

print("\n" + "=" * 60)
print("COUNT TABLE")
print(table)

# ==========================
# TARGET PER CELL
# ==========================
avg_samples = int(table.values.mean())

results = []

for length_group in table.index:
    for score_group in table.columns:

        count = table.loc[length_group, score_group]

        results.append({
            "length_group": length_group,
            "score_group": score_group,
            "count": count,
            "samples_to_add": max(0, avg_samples - count)
        })

results_df = pd.DataFrame(results)

# ==========================
# UNDERREPRESENTED GROUPS
# ==========================
missing = results_df[results_df["samples_to_add"] > 0]
missing = missing.sort_values("samples_to_add", ascending=False)

print("\n" + "=" * 60)
print(f"Average samples per group: {avg_samples}")

print("\nGROUPS THAT NEED MORE DATA")
print(missing)

# ==========================
# LENGTH LIMITS
# ==========================
print("\n" + "=" * 60)
print("LENGTH DEFINITIONS")

print(f"SHORT  : <= {int(q1)} words")
print(f"MEDIUM : {int(q1)+1} - {int(q2)} words")
print(f"LONG   : > {int(q2)} words")

# ==========================
# SCORE DISTRIBUTION
# ==========================
print("\n" + "=" * 60)
print("SCORE DISTRIBUTION")

print(
    df["score_group"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

TypeError: '<' not supported between instances of 'float' and 'str'

In [53]:


df["word_count"] = (
    df["student_answer"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

q1 = df["word_count"].quantile(0.33)
q2 = df["word_count"].quantile(0.66)

df["length_group"] = pd.cut(
    df["word_count"],
    bins=[-1, q1, q2, np.inf],
    labels=["short", "medium", "long"]
)

df["score_group"] = pd.cut(
    df["score"],
    bins=[-0.01, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=["0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]
)

table = pd.crosstab(
    df["length_group"],
    df["score_group"]
)

print(table)

TypeError: '<' not supported between instances of 'float' and 'str'

In [54]:

print("=" * 60)
print("QUESTION ANALYSIS")

num_questions = df["question"].nunique()
print(f"\nUnique questions: {num_questions}")

question_counts = df["question"].value_counts()

print("\nSamples per question:")
print(question_counts.describe())

print("\nTop 20 questions by sample count:")
print(question_counts.head(20))

print("\nBottom 20 questions by sample count:")
print(question_counts.tail(20))

print("\nQuestions with only 1 sample:")
print((question_counts == 1).sum())

print("\nQuestions with fewer than 5 samples:")
print((question_counts < 5).sum())

print("\nQuestions with fewer than 10 samples:")
print((question_counts < 10).sum())

QUESTION ANALYSIS

Unique questions: 3734

Samples per question:
count    3734.000000
mean        3.239154
std         8.883257
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max       233.000000
Name: count, dtype: float64

Top 20 questions by sample count:
question
מה היו הגורמים למלחמת העולם הראשונה?                                                            233
מהו המנגנון שמאפשר לתאים לייצר אנרגיה?                                                          191
כיצד השפיעו גילויי הארצות החדשות על העולם בתקופת העת החדשה? הסבר.                               152
איך מתרחש מחזור המים בטבע?                                                                      145
מדוע צמחים חשובים לקיום החיים על פני כדור הארץ? הסבר את תפקידם במערכת האקולוגית.                110
מה הייתה מלחמת העולם השנייה?                                                                    107
מה היו הגורמים המרכזיים למלחמת העולם השנייה וכיצד הם השפיעו על ההתפתחויות הבינלאומיות? הסבר.   

In [55]:


print("="*60)
print("TEACHER ANSWER ANALYSIS")

print("Unique teacher answers:")
print(df["teacher_answer"].nunique())

print("\nUnique (question, teacher_answer) pairs:")
print(
    df[["question","teacher_answer"]]
    .drop_duplicates()
    .shape[0]
)

teacher_counts = (
    df.groupby("teacher_answer")
    .size()
)

print("\nSamples per teacher answer:")
print(teacher_counts.describe())

TEACHER ANSWER ANALYSIS
Unique teacher answers:
4860

Unique (question, teacher_answer) pairs:
4922

Samples per teacher answer:
count    4860.000000
mean        2.488683
std         6.025506
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max       141.000000
dtype: float64


In [114]:


# =========================
# CONFIG
# =========================
SCORE_BINS = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
SCORE_LABELS = ["0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0"]

LENGTH_BINS = [0, 5, 9, 10**9]
LENGTH_LABELS = ["short", "medium", "long"]

TARGET_PER_GROUP = 200  # change this depending on dataset size


# =========================
# LOAD DATA
# =========================

# expected columns:
# question, teacher_answer, student_answer, score


# =========================
# FEATURES
# =========================
df["score_group"] = pd.cut(df["score"], bins=SCORE_BINS, labels=SCORE_LABELS, include_lowest=True)

df["answer_length"] = df["student_answer"].fillna("").apply(lambda x: len(x.split()))
df["length_group"] = pd.cut(df["answer_length"], bins=LENGTH_BINS, labels=LENGTH_LABELS)


# =========================
# GLOBAL STATISTICS
# =========================
print("\n========== GLOBAL DISTRIBUTION ==========")
global_table = pd.crosstab(df["length_group"], df["score_group"])
print(global_table)


# =========================
# QUESTION-LEVEL ANALYSIS
# =========================
results = []

for q, group in df.groupby("question"):

    pivot = pd.crosstab(group["length_group"], group["score_group"])

    for lg in LENGTH_LABELS:
        for sg in SCORE_LABELS:

            count = 0
            if lg in pivot.index and sg in pivot.columns:
                count = pivot.loc[lg, sg]

            results.append({
                "question": q,
                "length_group": lg,
                "score_group": sg,
                "count": count,
                "needs_more_data": count < TARGET_PER_GROUP
            })

report = pd.DataFrame(results)


# =========================
# PRIORITY ANALYSIS
# =========================
priority = (
    report[report["needs_more_data"]]
    .groupby(["length_group", "score_group"])
    .size()
    .reset_index(name="missing_questions")
    .sort_values("missing_questions", ascending=False)
)

print("\n========== MOST CRITICAL GAPS ==========")
print(priority.head(20))


# =========================
# QUESTIONS THAT ARE MOST IMBALANCED
# =========================
imbalance = []

for q, group in df.groupby("question"):
    dist = group["score_group"].value_counts(normalize=True)

    max_gap = dist.max() - dist.min()

    imbalance.append({
        "question": q,
        "samples": len(group),
        "imbalance_score": max_gap
    })

imbalance_df = pd.DataFrame(imbalance).sort_values("imbalance_score", ascending=False)

print("\n========== MOST BIASED QUESTIONS ==========")
print(imbalance_df.head(20))


# =========================
# SAVE OUTPUTS
# =========================
report.to_csv("per_question_gap_report.csv", index=False)
priority.to_csv("global_missing_groups.csv", index=False)
imbalance_df.to_csv("question_bias_report.csv", index=False)

print("\nSaved:")
print("- per_question_gap_report.csv")
print("- global_missing_groups.csv")
print("- question_bias_report.csv")


========== GLOBAL DISTRIBUTION ==========
score_group   0-0.2  0.2-0.4  0.4-0.6  0.6-0.8  0.8-1.0
length_group                                           
short           902      462      374      569     1991
medium          668      363      370      414     1366
long           1186      814      654      959     1630

========== MOST CRITICAL GAPS ==========
   length_group score_group  missing_questions
0          long       0-0.2               4063
1          long     0.2-0.4               4063
2          long     0.4-0.6               4063
3          long     0.6-0.8               4063
4          long     0.8-1.0               4063
5        medium       0-0.2               4063
6        medium     0.2-0.4               4063
7        medium     0.4-0.6               4063
8        medium     0.6-0.8               4063
9        medium     0.8-1.0               4063
10        short       0-0.2               4063
11        short     0.2-0.4               4063
12        short     0.4-

In [122]:


print("===== BASIC INFO =====")
print(df_model.info())
print(df_model.head())

# =========================
# 1. CHECK REQUIRED COLUMNS
# =========================
required_cols = {"text", "score"}

missing_cols = required_cols - set(df_model.columns)
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

print("\n✔ Columns OK")

# =========================
# 2. CHECK MISSING VALUES
# =========================
print("\n===== MISSING VALUES =====")
print(df_model.isnull().sum())

df_model = df_model.dropna().reset_index(drop=True)

# =========================
# 3. CHECK SCORE RANGE
# =========================
print("\n===== SCORE CHECK =====")

invalid_scores = df_model[(df_model["score"] < 0) | (df_model["score"] > 1)]
print("Invalid score rows:", len(invalid_scores))

if len(invalid_scores) > 0:
    print(invalid_scores.head())

# =========================
# 4. TEXT QUALITY CHECK
# =========================
print("\n===== TEXT CHECK =====")

df_model["text_len_chars"] = df_model["text"].astype(str).apply(len)
df_model["text_len_words"] = df_model["text"].astype(str).apply(lambda x: len(x.split()))

print(df_model[["text_len_chars", "text_len_words"]].describe())

empty_texts = df_model[df_model["text_len_words"] < 3]
print("Very short/empty texts:", len(empty_texts))

# =========================
# 5. DUPLICATES CHECK
# =========================
print("\n===== DUPLICATES =====")

dup = df_model.duplicated(subset=["text", "score"]).sum()
print("Duplicate rows:", dup)

# =========================
# 6. SCORE DISTRIBUTION
# =========================
print("\n===== SCORE DISTRIBUTION =====")

bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
df_model["score_bin"] = pd.cut(df_model["score"], bins=bins, include_lowest=True)

print(df_model["score_bin"].value_counts(normalize=True).sort_index() * 100)

# =========================
# 7. FINAL READINESS CHECK
# =========================
print("\n===== FINAL CHECK =====")

checks = {
    "has_data": len(df_model) > 1000,
    "no_missing_text": df_model["text"].isnull().sum() == 0,
    "valid_scores": len(invalid_scores) == 0,
    "no_tiny_texts": len(empty_texts) == 0
}

for k, v in checks.items():
    print(k, "✔" if v else "✘")

if all(checks.values()):
    print("\n✅ DATASET IS READY FOR TRAINING")
else:
    print("\n⚠ DATASET NEEDS FIXES BEFORE TRAINING")

===== BASIC INFO =====
<class 'pandas.DataFrame'>
RangeIndex: 12722 entries, 0 to 12721
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   text    12722 non-null  str    
 1   score   12722 non-null  float64
dtypes: float64(1), str(1)
memory usage: 4.3 MB
None
                                                text  score
0  [Q] מה היו תוצאות מלחמת העולם הראשונה? [T] סיו...    1.0
1  [Q] מה היו תוצאות מלחמת העולם הראשונה? [T] סיו...    0.9
2  [Q] מה היו תוצאות מלחמת העולם הראשונה? [T] סיו...    0.7
3  [Q] מה היו תוצאות מלחמת העולם הראשונה? [T] סיו...    0.4
4  [Q] מה היו תוצאות מלחמת העולם הראשונה? [T] סיו...    0.1

✔ Columns OK

===== MISSING VALUES =====
text     0
score    0
dtype: int64

===== SCORE CHECK =====
Invalid score rows: 0

===== TEXT CHECK =====
       text_len_chars  text_len_words
count    12722.000000    12722.000000
mean       193.019415       33.297280
std         84.961557       13.578387
min         41.0000

In [6]:
df_clean = df[["text", "score"]]

df_clean.to_csv("train.csv", index=False, encoding="utf-8-sig")


import os
print(os.getcwd())
print(os.listdir())

C:\git\CleverCheck\server\services
['check_answer_service.py', 'class_service.py', 'dataset.ipynb', 'dataset.jsonl', 'exam_service.py', 'global_missing_groups.csv', 'grading_service.py', 'option_service.py', 'per_question_gap_report.csv', 'question_bias_report.csv', 'question_service.py', 'question_type_service.py', 'student_answer_service.py', 'student_exam_service.py', 'student_service.py', 'subject_service.py', 'Synonym_reverso.py', 'teacher_answer_service.py', 'teacher_service.py', 'text_processing_service.py', 'train.csv', 'try.py', 'try2.py', 'try3.py', 'try4.py', 'try6.py', 'try_my_model.py', '__init__.py', '__pycache__']
